In [1]:
%pip install xgboost imbalanced-learn scikit-learn pandas numpy matplotlib seaborn joblib lightgbm optuna

Note: you may need to restart the kernel to use updated packages.


### Data Preparation

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.impute import KNNImputer, SimpleImputer
from imblearn.over_sampling import SMOTE
import xgboost as xgb
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

In [3]:
hot_rolling_df = pd.read_csv('dataset/train.csv')
print(f"Hot Rolling Dataset Shape : {hot_rolling_df.shape}")
hot_rolling_df.head(20)

Hot Rolling Dataset Shape : (1352, 51)


,CoilID,X1,X2,X3,X4,X5,X6,X7,X8,X9,...,X41,X42,X43,X44,X45,X46,X47,X48,X49,Y
0,487,854.787195,501.088868,414.841484,710.583316,662.072013,656.076977,547.040479,563.653582,495.296785,...,0.201645,0.047960,0.267467,0.052247,-0.893174,0.000000,0.028925,0.000534,0.010797,0.0
1,44,1056.526699,868.083321,622.879982,725.276469,665.235554,647.450550,552.333202,565.105074,493.310075,...,0.644403,0.000000,0.341870,0.153513,25.471899,0.002520,0.033281,0.028349,0.079602,0.0
2,192,1095.648362,668.112517,695.787904,716.773671,662.843475,657.542380,549.863867,546.210823,482.814753,...,0.486502,0.000000,0.202539,0.168192,-25.764196,0.002072,0.033878,0.000000,0.058266,0.0
3,1552,1050.943543,660.340015,440.280245,611.562496,628.081103,561.397721,456.816210,550.103433,378.353283,...,1.198010,0.020787,0.288786,0.329108,1.033840,0.000250,0.045490,0.039004,0.004850,0.0
4,1190,1091.640314,297.363775,842.665620,749.160886,652.992309,615.576656,608.364764,549.756758,487.753140,...,0.237231,0.000841,0.257281,0.112637,-11.130157,0.002376,0.031298,0.003623,0.018434,0.0
5,102,1060.742584,809.756675,675.882581,726.410383,672.159859,661.250928,555.468859,554.440758,492.462056,...,0.810914,0.043039,0.274467,0.250703,28.221606,0.002755,0.049610,0.001055,0.117253,0.0
6,900,1087.182781,513.823963,434.777529,727.732813,661.179522,606.946096,539.393969,532.864699,480.662740,...,0.565355,0.000846,0.341627,0.115135,8.942980,0.002918,0.051415,0.029154,0.084616,0.0
7,674,1082.185852,328.168651,395.732261,610.456744,650.491924,532.674335,467.587611,544.260627,486.572833,...,0.198847,0.001384,0.256416,0.042418,21.992259,0.000704,0.039402,0.000028,0.009336,0.0
8,572,1111.579895,909.156996,317.875943,731.479162,672.679098,663.639612,547.616420,571.662343,490.837575,...,0.518436,0.000000,0.215945,0.051108,-17.106723,0.000842,0.037112,0.002238,0.011166,0.0
9,1205,1037.383965,609.839905,513.596340,728.363034,645.250990,590.713451,601.842899,520.893575,484.417182,...,1.010123,0.000000,0.231285,0.010257,6.860036,0.002862,0.047626,0.032069,0.119772,0.0


In [4]:
# Check for the missing values in the before starting to 'impute' in the  dataset
missing_values = hot_rolling_df.isnull().sum()
print("Missing values in each of the column features in the dataset : \n", missing_values)
hot_rolling_df.info()
hot_rolling_df.describe()

Missing values in each of the column features in the dataset : 
 CoilID      0
X1          0
X2          0
X3          0
X4          0
X5          0
X6          0
X7          0
X8          1
X9          0
X10         6
X11         0
X12         0
X13         0
X14         0
X15       160
X16         6
X17         0
X18         0
X19         0
X20         0
X21         1
X22         0
X23         6
X24         6
X25         6
X26         7
X27         6
X28         0
X29         0
X30         0
X31         0
X32         0
X33         0
X34         0
X35         0
X36         0
X37         0
X38         0
X39         0
X40         0
X41         0
X42        31
X43         0
X44         0
X45         0
X46         0
X47         0
X48        13
X49         0
Y           0
dtype: int64
<class 'pandas.DataFrame'>
RangeIndex: 1352 entries, 0 to 1351
Data columns (total 51 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   CoilID  1352 non-null   int64  
 

,CoilID,X1,X2,X3,X4,X5,X6,X7,X8,X9,...,X41,X42,X43,X44,X45,X46,X47,X48,X49,Y
count,1352.000000,1352.000000,1352.000000,1352.000000,1352.000000,1352.000000,1352.000000,1352.000000,1351.000000,1352.000000,...,1352.000000,1321.000000,1352.000000,1352.000000,1352.000000,1352.000000,1352.000000,1339.000000,1352.000000,1352.000000
mean,833.515533,1028.915322,575.374555,538.372787,692.576938,649.010832,618.584573,529.241056,528.770054,461.699867,...,0.585886,0.011334,0.281290,0.091495,11.507361,0.002745,0.050913,0.007108,0.064400,0.048817
std,487.420566,108.505011,232.103882,135.288085,56.798904,35.398207,45.602831,46.263179,40.220201,40.600916,...,0.296115,0.017114,0.068455,0.073058,25.004864,0.006660,0.044617,0.012504,0.050905,0.215564
min,1.000000,235.252250,96.755492,124.150450,575.916250,559.272859,529.937396,439.221384,425.413251,343.110465,...,0.077352,0.000000,0.029599,0.000000,-82.672877,0.000000,0.016833,0.000000,0.000000,0.000000
25%,411.250000,1009.279089,405.533242,441.585514,622.213663,625.327743,583.363796,476.813303,520.192858,423.604248,...,0.365881,0.000380,0.244377,0.026073,-1.368777,0.000943,0.039864,0.000000,0.015624,0.000000
50%,824.500000,1071.978233,589.160841,548.035306,724.970407,661.170690,615.240491,547.648095,545.399528,482.412285,...,0.574328,0.002613,0.293164,0.076108,9.177752,0.001644,0.045829,0.001056,0.060336,0.000000
75%,1254.250000,1092.031100,725.766924,632.238363,734.453213,668.054192,659.564239,554.584166,556.334294,488.713960,...,0.797025,0.013933,0.334613,0.143156,24.196605,0.002275,0.051435,0.002679,0.106017,0.000000
max,1691.000000,1124.903234,1148.171484,1026.915778,755.983296,763.257466,742.725523,618.947910,575.312130,505.349388,...,1.377925,0.062637,0.409739,0.329108,120.169658,0.064545,0.424405,0.051150,0.249522,1.000000


In [5]:
# Separating the features and the target variable 'Y'

X_hot_rolling = hot_rolling_df.drop(columns=['CoilID', 'Y'])
Y_hot_rolling = hot_rolling_df['Y']


In [6]:
# Handling the missing values using the Imputer (KNNImputer)

imputer = KNNImputer(n_neighbors=5)
X_hot_rolling_imputed = imputer.fit_transform(X_hot_rolling)
# converting it to the dataframe
X_hot_rolling_imputed = pd.DataFrame(X_hot_rolling_imputed, columns=X_hot_rolling.columns)


# checking the missing values after imputation
print("Missing values after imputation: ")
print(X_hot_rolling_imputed.isnull().sum())
print("Shape of the imputed dataset: ", X_hot_rolling_imputed.shape)

hot_rolling_df.info()
hot_rolling_df.describe()


Missing values after imputation: 
X1     0
X2     0
X3     0
X4     0
X5     0
X6     0
X7     0
X8     0
X9     0
X10    0
X11    0
X12    0
X13    0
X14    0
X15    0
X16    0
X17    0
X18    0
X19    0
X20    0
X21    0
X22    0
X23    0
X24    0
X25    0
X26    0
X27    0
X28    0
X29    0
X30    0
X31    0
X32    0
X33    0
X34    0
X35    0
X36    0
X37    0
X38    0
X39    0
X40    0
X41    0
X42    0
X43    0
X44    0
X45    0
X46    0
X47    0
X48    0
X49    0
dtype: int64
Shape of the imputed dataset:  (1352, 49)
<class 'pandas.DataFrame'>
RangeIndex: 1352 entries, 0 to 1351
Data columns (total 51 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   CoilID  1352 non-null   int64  
 1   X1      1352 non-null   float64
 2   X2      1352 non-null   float64
 3   X3      1352 non-null   float64
 4   X4      1352 non-null   float64
 5   X5      1352 non-null   float64
 6   X6      1352 non-null   float64
 7   X7      1352 non-null   float64
 8  

,CoilID,X1,X2,X3,X4,X5,X6,X7,X8,X9,...,X41,X42,X43,X44,X45,X46,X47,X48,X49,Y
count,1352.000000,1352.000000,1352.000000,1352.000000,1352.000000,1352.000000,1352.000000,1352.000000,1351.000000,1352.000000,...,1352.000000,1321.000000,1352.000000,1352.000000,1352.000000,1352.000000,1352.000000,1339.000000,1352.000000,1352.000000
mean,833.515533,1028.915322,575.374555,538.372787,692.576938,649.010832,618.584573,529.241056,528.770054,461.699867,...,0.585886,0.011334,0.281290,0.091495,11.507361,0.002745,0.050913,0.007108,0.064400,0.048817
std,487.420566,108.505011,232.103882,135.288085,56.798904,35.398207,45.602831,46.263179,40.220201,40.600916,...,0.296115,0.017114,0.068455,0.073058,25.004864,0.006660,0.044617,0.012504,0.050905,0.215564
min,1.000000,235.252250,96.755492,124.150450,575.916250,559.272859,529.937396,439.221384,425.413251,343.110465,...,0.077352,0.000000,0.029599,0.000000,-82.672877,0.000000,0.016833,0.000000,0.000000,0.000000
25%,411.250000,1009.279089,405.533242,441.585514,622.213663,625.327743,583.363796,476.813303,520.192858,423.604248,...,0.365881,0.000380,0.244377,0.026073,-1.368777,0.000943,0.039864,0.000000,0.015624,0.000000
50%,824.500000,1071.978233,589.160841,548.035306,724.970407,661.170690,615.240491,547.648095,545.399528,482.412285,...,0.574328,0.002613,0.293164,0.076108,9.177752,0.001644,0.045829,0.001056,0.060336,0.000000
75%,1254.250000,1092.031100,725.766924,632.238363,734.453213,668.054192,659.564239,554.584166,556.334294,488.713960,...,0.797025,0.013933,0.334613,0.143156,24.196605,0.002275,0.051435,0.002679,0.106017,0.000000
max,1691.000000,1124.903234,1148.171484,1026.915778,755.983296,763.257466,742.725523,618.947910,575.312130,505.349388,...,1.377925,0.062637,0.409739,0.329108,120.169658,0.064545,0.424405,0.051150,0.249522,1.000000


In [7]:
# Feature Scaling or Normalization of the dataset

scaler = StandardScaler()
X_hot_rolling_scaled = scaler.fit_transform(X_hot_rolling_imputed)

In [8]:
# Feature Scaling 

from sklearn.feature_selection import SelectFromModel, mutual_info_classif

mi_scores_hot_rolling = mutual_info_classif(X_hot_rolling_scaled, Y_hot_rolling, random_state=42)
mi_series_hot_rolling = pd.Series(mi_scores_hot_rolling, index=X_hot_rolling.columns).sort_values(ascending=False)

print("Top 10 features based on the Mutual Information Scores: \n", mi_series_hot_rolling.head(10))



Top 10 features based on the Mutual Information Scores: 
 X36    0.039463
X13    0.038971
X34    0.033676
X32    0.033534
X39    0.030829
X10    0.028052
X30    0.027061
X6     0.026641
X35    0.026247
X33    0.024256
dtype: float64


In [9]:
# Selecting the top 20 features 

top_20_features_hot_rolling = mi_series_hot_rolling.head(20).index.tolist()

X_hot_rolling_selected = X_hot_rolling_imputed[top_20_features_hot_rolling]

print("shape of the dataset after the feature selection: ", X_hot_rolling_selected.shape)

X_hot_rolling_selected.describe()


shape of the dataset after the feature selection:  (1352, 20)


,X36,X13,X34,X32,X39,X10,X30,X6,X35,X33,X31,X28,X18,X15,X38,X29,X37,X5,X41,X24
count,1352.000000,1352.000000,1352.000000,1352.000000,1352.000000,1352.000000,1352.000000,1352.000000,1.352000e+03,1352.000000,1352.000000,1352.000000,1352.000000,1352.000000,1352.000000,1352.000000,1352.000000,1352.000000,1352.000000,1352.000000
mean,2279.289201,868.915869,2359.427515,15.998457,162.281065,6.861327,10.095567,618.584573,9.922451e+06,17.119898,13.506972,4.749715,890.539824,3.446813,690.497041,6.948415,1641.233728,649.010832,0.585886,16.349268
std,1767.389750,397.296095,1770.224468,3.784762,11.368766,2.631377,2.250786,45.602831,6.665497e+06,3.670994,3.259872,0.602077,7.991223,2.172425,1263.545639,1.093988,1741.161108,35.398207,0.296115,4.990185
min,0.000000,97.078178,0.000000,6.508389,98.000000,1.117275,4.356228,529.937396,0.000000e+00,6.714073,5.880277,4.321103,858.748809,1.173575,0.000000,4.578510,0.000000,559.272859,0.077352,-1.262651
25%,98.750000,545.825307,90.000000,13.470451,160.000000,4.702552,8.769181,583.363796,4.627258e+05,14.923421,11.121122,4.482384,887.174583,2.005021,0.000000,6.234717,45.000000,625.327743,0.365881,14.711926
50%,3559.500000,842.055177,3487.000000,16.295053,164.000000,6.983717,10.119274,615.240491,1.394034e+07,19.046641,13.846186,4.543375,890.031317,2.776068,74.500000,6.912886,553.000000,661.170690,0.574328,16.942795
75%,3865.250000,1169.875976,3861.250000,19.721242,169.000000,8.940686,11.774199,659.564239,1.488926e+07,19.977972,15.900814,4.646584,896.489977,4.089900,613.750000,7.834167,3756.250000,668.054192,0.797025,19.368358
max,4296.000000,1631.918408,4380.000000,24.508596,173.000000,12.312371,14.913641,742.725523,1.774067e+07,25.232161,20.290997,6.635675,918.084739,12.327319,4155.000000,10.253523,4457.000000,763.257466,1.377925,30.789863


In [10]:
# Adressing the class imbalance using SMOTE

stratified_shuffled_split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

for train_indx, test_indx in stratified_shuffled_split.split(X_hot_rolling_selected, Y_hot_rolling):
    
    X_train_hot_rolling, X_test_hot_rolling = X_hot_rolling_selected.iloc[train_indx], X_hot_rolling_selected.iloc[test_indx]
    y_train_hot_rolling, y_test_hot_rolling = Y_hot_rolling[train_indx], Y_hot_rolling[test_indx]
    

print(f"Train defect ratio : {y_train_hot_rolling.mean():.4f}")
print(f"Test defect ratio : {y_test_hot_rolling.mean():.4f}")

# Applying SMOTE to the training data

smote = SMOTE(random_state=42, sampling_strategy=0.3)
X_train_hot_rolling_smote, y_train_hot_rolling_smote = smote.fit_resample(X_train_hot_rolling, y_train_hot_rolling)

print(f"Shape of training data after SMOTE: {X_train_hot_rolling_smote.shape}")
print(f"Defect ratio of training data after SMOTE: {y_train_hot_rolling_smote.mean():.4f}")
    


Train defect ratio : 0.0490
Test defect ratio : 0.0480
Shape of training data after SMOTE: (1336, 20)
Defect ratio of training data after SMOTE: 0.2305


### Model Preparation

In [11]:
# calculating scale_pos_weight as the ratio of the negative to positive samples

from sklearn.metrics import recall_score, precision_score, f1_score, roc_auc_score, precision_recall_curve, confusion_matrix

scale_pos_weight = (y_train_hot_rolling == 0).sum() / (y_train_hot_rolling == 1).sum()
print(f"Scale pos weight : {scale_pos_weight:.4f}")


xgb_classifier = xgb.XGBClassifier(
    scale_pos_weight = scale_pos_weight,
    eval_metric = 'logloss',
    random_state = 42,
    use_label_encoder = False
)

xgb_classifier.fit(X_train_hot_rolling_smote, y_train_hot_rolling_smote)

# Evaluating on the test set
y_pred_xgb = xgb_classifier.predict(X_test_hot_rolling)

print("Baseline XGBoost Classifier Performance : ")
print(f"Recall : {recall_score(y_test_hot_rolling, y_pred_xgb):.4f}")
print(f"Precision : {precision_score(y_test_hot_rolling, y_pred_xgb):.4f}")
print(f"F1 Score : {f1_score(y_test_hot_rolling, y_pred_xgb):.4f}")
print(f"ROC AUC Score : {roc_auc_score(y_test_hot_rolling, y_pred_xgb):.4f}")



Scale pos weight : 19.3962
Baseline XGBoost Classifier Performance : 
Recall : 0.4615
Precision : 0.2609
F1 Score : 0.3333
ROC AUC Score : 0.6978


In [12]:
## Hyperparameter Tuning using GridSearchCV

import optuna
from sklearn.model_selection import GridSearchCV, cross_val_predict

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 300, 800, step=50),
        'max_depth': trial.suggest_int('max_depth', 4, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 0.9),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 0.9),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma': trial.suggest_float('gamma', 0, 0.5),
        'reg_alpha': trial.suggest_float('reg_alpha', 0, 0.1),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.5, 2.0),
        'scale_pos_weight': trial.suggest_int('scale_pos_weight', 50, 150),
        'eval_metric': 'logloss',
        'random_state': 42,
        'use_label_encoder': False
    }
    
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    recall_scores = []
    precision_scores = []
    
    for train_indx, val_indx in skf.split(X_train_hot_rolling_smote, y_train_hot_rolling_smote):
        X_train_fold, X_val_fold = X_train_hot_rolling_smote.iloc[train_indx], X_train_hot_rolling_smote.iloc[val_indx]
        y_train_fold, y_val_fold = y_train_hot_rolling_smote.iloc[train_indx], y_train_hot_rolling_smote.iloc[val_indx]
        
        model = xgb.XGBClassifier(**params)
        model.fit(X_train_fold, y_train_fold)
        
        y_val_pred = model.predict(X_val_fold)
        recall_scores.append(recall_score(y_val_fold, y_val_pred))
        precision_scores.append(precision_score(y_val_fold, y_val_pred))
        
        # using f2 score as the optimization metric to give more weight to recall
        
        f2_scores = (5 * np.mean(recall_scores) * np.mean(precision_scores)) / (4 * np.mean(precision_scores) + np.mean(recall_scores) + 1e-8)
        
        return f2_scores
        


In [13]:
%pip install --upgrade xgboost

Note: you may need to restart the kernel to use updated packages.


In [14]:
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=50, show_progress_bar=True)
best_params = study.best_params
print("Best Hyperparameters : ", best_params)

# Training final XGBoost model with the best hyperparameters
xgb_final_classifier = xgb.XGBClassifier(**best_params, use_label_encoder=False)
xgb_final_classifier.fit(X_train_hot_rolling_smote, y_train_hot_rolling_smote)


# Cross validation with best model (Stratified K-Fold)


skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_recall = []
cv_precision = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_hot_rolling_smote, y_train_hot_rolling_smote)):
    X_train_fold, X_val_fold = X_train_hot_rolling_smote.iloc[train_idx], X_train_hot_rolling_smote.iloc[val_idx]
    y_train_fold, y_val_fold = y_train_hot_rolling_smote.iloc[train_idx], y_train_hot_rolling_smote.iloc[val_idx]
    
    model = xgb.XGBClassifier(**best_params, use_label_encoder=False)
    model.fit(X_train_fold, y_train_fold)
    
    y_pred = model.predict(X_val_fold)
    cv_recall.append(recall_score(y_val_fold, y_pred))
    cv_precision.append(precision_score(y_val_fold, y_pred))
    
    print(f"Fold {fold+1} -  Recall: {cv_recall[-1]:.4f}, Precision: {cv_precision[-1]:.4f}")
    
print(f"CV Average Recall: {np.mean(cv_recall):.4f} (+/- {np.std(cv_recall):.4f})")
print(f"CV Average Precision: {np.mean(cv_precision):.4f} (+/- {np.std(cv_precision):.4f})")
    
    

[I 2026-05-30 20:45:36,545] A new study created in memory with name: no-name-d2cb01da-7b95-4c07-9a65-366a881bd4be
Best trial: 0. Best value: 0.885036:   2%|▏         | 1/50 [00:00<00:43,  1.14it/s]

[I 2026-05-30 20:45:37,429] Trial 0 finished with value: 0.8850364940180617 and parameters: {'n_estimators': 650, 'max_depth': 6, 'learning_rate': 0.012679121823787368, 'subsample': 0.7823185623925358, 'colsample_bytree': 0.7043040584650825, 'min_child_weight': 6, 'gamma': 0.35108915219112624, 'reg_alpha': 0.06783824318373725, 'reg_lambda': 1.3776293608097994, 'scale_pos_weight': 98}. Best is trial 0 with value: 0.8850364940180617.


Best trial: 0. Best value: 0.885036:   4%|▍         | 2/50 [00:01<00:28,  1.69it/s]

[I 2026-05-30 20:45:37,821] Trial 1 finished with value: 0.8874045780519784 and parameters: {'n_estimators': 600, 'max_depth': 7, 'learning_rate': 0.06722086919791677, 'subsample': 0.6289892345337543, 'colsample_bytree': 0.7088871271545676, 'min_child_weight': 1, 'gamma': 0.2772170707853972, 'reg_alpha': 0.09673635635876669, 'reg_lambda': 1.6184682089505251, 'scale_pos_weight': 61}. Best is trial 0 with value: 0.8850364940180617.


Best trial: 2. Best value: 0.880682:   6%|▌         | 3/50 [00:01<00:23,  1.98it/s]

...

[I 2026-05-30 20:45:55,470] Trial 47 finished with value: 0.8225806423746099 and parameters: {'n_estimators': 300, 'max_depth': 4, 'learning_rate': 0.011937625750663311, 'subsample': 0.6412165714781606, 'colsample_bytree': 0.7581189011505405, 'min_child_weight': 10, 'gamma': 0.43563642580859496, 'reg_alpha': 0.022779449450649424, 'reg_lambda': 1.6883088486660718, 'scale_pos_weight': 124}. Best is trial 22 with value: 0.814696482810634.


Best trial: 22. Best value: 0.814696:  98%|█████████▊| 49/50 [00:19<00:00,  3.31it/s]

[I 2026-05-30 20:45:55,706] Trial 48 finished with value: 0.8629441597972694 and parameters: {'n_estimators': 300, 'max_depth': 4, 'learning_rate': 0.017936084367096285, 'subsample': 0.6388594847584613, 'colsample_bytree': 0.7378782836696735, 'min_child_weight': 10, 'gamma': 0.4394296343868231, 'reg_alpha': 0.023556456756210164, 'reg_lambda': 1.3909340454215247, 'scale_pos_weight': 124}. Best is trial 22 with value: 0.814696482810634.


Best trial: 22. Best value: 0.814696: 100%|██████████| 50/50 [00:19<00:00,  2.57it/s]


[I 2026-05-30 20:45:56,009] Trial 49 finished with value: 0.8388157867430965 and parameters: {'n_estimators': 300, 'max_depth': 5, 'learning_rate': 0.01244678106775338, 'subsample': 0.6063673908879321, 'colsample_bytree': 0.7652111040185318, 'min_child_weight': 10, 'gamma': 0.47649577756966377, 'reg_alpha': 0.0057321852893319515, 'reg_lambda': 1.6323645051400117, 'scale_pos_weight': 138}. Best is trial 22 with value: 0.814696482810634.
Best Hyperparameters :  {'n_estimators': 300, 'max_depth': 4, 'learning_rate': 0.010574897310246738, 'subsample': 0.6222092160980853, 'colsample_bytree': 0.6775731008153905, 'min_child_weight': 8, 'gamma': 0.44079853835650235, 'reg_alpha': 0.07648611346596619, 'reg_lambda': 1.997057108178758, 'scale_pos_weight': 109}
Fold 1 -  Recall: 0.9839, Precision: 0.5126
Fold 2 -  Recall: 1.0000, Precision: 0.4357
Fold 3 -  Recall: 0.9836, Precision: 0.4651
Fold 4 -  Recall: 1.0000, Precision: 0.4397
Fold 5 -  Recall: 1.0000, Precision: 0.4161
CV Average Recall: 0.

### Ensemble: LightGBM + Logistics Regression

In [15]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import VotingClassifier, StackingClassifier

lgb_model = lgb.LGBMClassifier(
    is_unbalance=True,
    learning_rate=0.05,
    n_estimators=500,
    max_depth=5,
    num_leaves=31,
    random_state=42
)

lgb_model.fit(X_train_hot_rolling_smote, y_train_hot_rolling_smote)

# Training a simple logistic regression model for comparision (with SMOTE data)
log_reg_model = LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)
log_reg_model.fit(X_train_hot_rolling_smote, y_train_hot_rolling_smote)


# Soft voting ensemble of XGBoost, LightGBM and Logistic Regression
voting_clf = VotingClassifier(
    estimators=[
        ('xgb', xgb_final_classifier),
        ('lgb', lgb_model),
        ('log_reg', log_reg_model)
    ],
    voting='soft',
    weights=[1.5, 1.2, 0.8]
)

voting_clf.fit(X_train_hot_rolling_smote, y_train_hot_rolling_smote)

# Stacking ensemble using the predictions of the base models which is used as the features
# for the meta model (Logistic Regression)

stacking_clf = StackingClassifier(
    estimators=[
        ('xgb', xgb_final_classifier),
        ('lgb', lgb_model)
    ],
    final_estimator=LogisticRegression(C=1.0),
    cv=5,
    stack_method='predict_proba'
)

stacking_clf.fit(X_train_hot_rolling_smote, y_train_hot_rolling_smote)


# Evaluating both the ensemble models on test set with threshold 0.5

for name, model in [('Voting', voting_clf), ('Stacking', stacking_clf)]:
    y_pred = model.predict(X_test_hot_rolling)
    print(f"\n{name} Ensemble Performance : ")
    print(f"Recall : {recall_score(y_test_hot_rolling, y_pred):.4f}")
    print(f"Precision : {precision_score(y_test_hot_rolling, y_pred):.4f}")





[LightGBM] [Info] Number of positive: 308, number of negative: 1028
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000207 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5099
[LightGBM] [Info] Number of data points in the train set: 1336, number of used features: 20
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.230539 -> initscore=-1.205271
[LightGBM] [Info] Start training from score -1.205271
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, b

### Evaluation - Achieving 100% Recall and >90% Accuracy

In [16]:
def evaluate_model(model, X_test, y_test, threshold=0.5, model_name="Model"):
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    y_pred = (y_pred_proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    recall = tp / (tp + fn) if (tp+fn) > 0 else 0
    precision = tp / (tp + fp) if (tp+fp) > 0 else 0
    print(f"\n{model_name} - Threshold {threshold:.2f}")
    print(f"Confusion Matrix: TP={tp}, FP={fp}, FN={fn}, TN={tn}")
    print(f"Recall: {recall:.4f}  (Target: 1.0000)")
    print(f"Precision: {precision:.4f} (Target: >0.90)")
    return recall, precision, y_pred_proba


best_model = stacking_clf
evaluate_model(best_model, X_test_hot_rolling, y_test_hot_rolling, threshold=0.5)
    


Model - Threshold 0.50
Confusion Matrix: TP=4, FP=14, FN=9, TN=244
Recall: 0.3077  (Target: 1.0000)
Precision: 0.2222 (Target: >0.90)


(np.float64(0.3076923076923077),
 np.float64(0.2222222222222222),
 array([0.0073686 , 0.00737687, 0.00718616, 0.00809105, 0.01772461,
        0.11323958, 0.00700309, 0.00759376, 0.00758373, 0.00708978,
        0.8238935 , 0.03541479, 0.00743323, 0.00877205, 0.00730356,
        0.00748442, 0.00898143, 0.01938857, 0.00708784, 0.00846305,
        0.06047031, 0.875695  , 0.01122821, 0.00725583, 0.3131715 ,
        0.05161468, 0.78009033, 0.00718322, 0.11329924, 0.06130156,
        0.17154721, 0.00718388, 0.00731097, 0.00763854, 0.80818957,
        0.06007354, 0.00746108, 0.00822369, 0.08728204, 0.00914807,
        0.00771628, 0.00823401, 0.00716604, 0.03754698, 0.06125808,
        0.03446799, 0.78578983, 0.00724288, 0.00737813, 0.07448964,
        0.06222739, 0.04624693, 0.07253249, 0.02024354, 0.01379533,
        0.01916203, 0.06071588, 0.00742193, 0.82414707, 0.02723923,
        0.39015555, 0.13132257, 0.00745585, 0.0567231 , 0.00713594,
        0.00719123, 0.20104639, 0.00712613, 0.0120

In [17]:
# Threshold tuning using Precision-Recall curve

from sklearn.model_selection import train_test_split

def find_optimal_threshold(model, X_val, y_val, target_recall=1.0, target_precision=0.9):
    y_proba = model.predict_proba(X_val)[:, 1]
    precisions, recalls, thresholds = precision_recall_curve(y_val, y_proba)
    # We need the highest threshold where recall >= target_recall and precision >= target_precision
    feasible = []
    for i in range(len(thresholds)):
        if recalls[i] >= target_recall - 1e-6 and precisions[i] >= target_precision - 1e-6:
            feasible.append((thresholds[i], recalls[i], precisions[i]))
    if feasible:
        # Among feasible, choose threshold that maximizes precision (or simply the one with highest precision)
        best = max(feasible, key=lambda x: x[2])   # highest precision
        return best[0], best[1], best[2]
    else:
        # No threshold satisfies both; find threshold that gives recall=1.0 with best precision
        recall_1_indices = [i for i, r in enumerate(recalls) if r >= target_recall - 1e-6]
        if recall_1_indices:
            best_idx = max(recall_1_indices, key=lambda i: precisions[i])
            return thresholds[best_idx], recalls[best_idx], precisions[best_idx]
        else:
            return 0.5, recalls[-1], precisions[-1]
        
        
X_train_final, X_val_thresh, y_train_final, y_val_thresh = train_test_split(
    X_train_hot_rolling_smote, 
    y_train_hot_rolling_smote, 
    test_size=0.2,
    stratify=y_train_hot_rolling_smote,
    random_state=42
)

# Retrain best model on X_train_final
final_model = stacking_clf 
final_model.fit(X_train_final, y_train_final)

opt_threshold, opt_recall, opt_precision = find_optimal_threshold(
    final_model, X_val_thresh, y_val_thresh, target_recall=1.0, target_precision=0.9
)
print(f"\nOptimal threshold: {opt_threshold:.3f} (Recall={opt_recall:.4f}, Precision={opt_precision:.4f})")


# Evaluate on test set with optimal threshold
_, _, y_proba_test = evaluate_model(final_model, X_test_hot_rolling, y_test_hot_rolling, threshold=opt_threshold, model_name="Final Model with Optimal Threshold")



[LightGBM] [Info] Number of positive: 246, number of negative: 822
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000219 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4929
[LightGBM] [Info] Number of data points in the train set: 1068, number of used features: 20
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.230337 -> initscore=-1.206409
[LightGBM] [Info] Start training from score -1.206409
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, be

In [18]:

# Final Check: Guarantee Recall = 100% on test?

# If recall is still < 100%, we can try lowering threshold further until recall=1.0
def threshold_for_recall_1(model, X_val, y_val):
    y_proba = model.predict_proba(X_val)[:, 1]
    precisions, recalls, thresholds = precision_recall_curve(y_val, y_proba)
    for i in range(len(thresholds)):
        if recalls[i] >= 1.0 - 1e-6:
            return thresholds[i], precisions[i]
    return 0.1, precisions[-1]   # fallback

thresh_recall1, prec_at_thresh = threshold_for_recall_1(final_model, X_val_thresh, y_val_thresh)
print(f"\nThreshold for 100% recall: {thresh_recall1:.3f} (Precision = {prec_at_thresh:.4f})")

# Evaluate test set with that threshold
_, _, _ = evaluate_model(final_model, X_test_hot_rolling, y_test_hot_rolling, threshold=thresh_recall1, model_name="100% Recall Threshold")


Threshold for 100% recall: 0.009 (Precision = 0.2313)

100% Recall Threshold - Threshold 0.01
Confusion Matrix: TP=13, FP=258, FN=0, TN=0
Recall: 1.0000  (Target: 1.0000)
Precision: 0.0480 (Target: >0.90)


In [20]:
import joblib
joblib.dump(final_model, 'alpha_defect_model.pkl')
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(imputer, 'imputer.pkl')
joblib.dump(top_20_features_hot_rolling, 'selected_features.pkl')

['selected_features.pkl']